notebook_03_normalization.ipynb

Phase 3 — Normalization

Input:  product_entities_v2.json

Output: product_entities_normalized_v2.json

product_entities_normalized_v2.csv

No API calls in this notebook — pure Python, completely free to run.

In [1]:
import json
import pandas as pd
from copy import deepcopy

print('No API calls in this notebook — runs free.')

No API calls in this notebook — runs free.


In [2]:
# ── CELL 2 — Load entities ───────────────────────────────────────────────────
from google.colab import files
uploaded = files.upload()   # upload product_entities_v2.json

with open('product_entities_v2.json', encoding='utf-8') as f:
    data = json.load(f)

print(f'Loaded: {len(data)} products')
print(f'Keys:   {list(data[0].keys())}')


Saving product_entities_v2.json to product_entities_v2.json
Loaded: 114 products
Keys:   ['product_id', 'product_name_cn', 'product_name', 'target_crops', 'target_diseases', 'target_pests', 'symptoms', 'active_ingredients', 'product_types', 'dosages', 'application_methods', 'safety_warnings']


In [3]:
# ── CELL 3 — Normalization helper ────────────────────────────────────────────
def normalize_list(values, canonical_map):
    """
    Normalize a list of string values:
    1. Strip whitespace
    2. Apply canonical_map (case-insensitive lookup)
    3. Remove duplicates preserving order
    4. Remove empty strings
    """
    seen   = set()
    result = []
    for v in values:
        v = v.strip()
        if not v:
            continue
        # Case-insensitive lookup in canonical map
        canonical = canonical_map.get(v.lower(), v)
        if canonical.lower() not in seen:
            seen.add(canonical.lower())
            result.append(canonical)
    return result

In [10]:
# ── CELL 4 — Normalize: target_crops ─────────────────────────────────────────
# Rule: Title Case for all crop names
# 62 case variants found — fix by applying Title Case as the canonical form

CROP_MAP = {}  # built dynamically — Title Case wins

def normalize_crops(values):
    seen   = set()
    result = []
    for v in values:
        v = v.strip()
        if not v:
            continue
        # Title case each word
        canonical = v.title()
        # Exceptions: keep known acronyms/special cases
        canonical = canonical.replace("And", "and").replace("Or", "or").replace(
            "Of", "of").replace("For", "for").replace("In", "in")
        # Fix first word if it got lowercased by exceptions
        if canonical and canonical[0].islower():
            canonical = canonical[0].upper() + canonical[1:]
        if canonical.lower() not in seen:
            seen.add(canonical.lower())
            result.append(canonical)
    return result

for record in data:
    record['target_crops'] = normalize_crops(record.get('target_crops', []))

# Verify
sample = next(r for r in data if r['product_id'] == 'AF0001')
print('AF0001 crops sample:', sample['target_crops'][:5])

AF0001 crops sample: ['Chili Pepper', 'Citrus', 'Vegetables', 'Fruit Trees', 'Melons']


In [20]:
# ── CELL 5 — Normalize: target_diseases ──────────────────────────────────────
# Rule: Title Case — "downy mildew" → "Downy Mildew"

# Fix normalize_diseases to use Title Case consistently
def normalize_diseases(values):
    seen   = set()
    result = []
    for v in values:
        v = v.strip()
        if not v:
            continue
        # Title Case — "downy mildew" → "Downy Mildew"
        canonical = v.title()
        # Fix connector words
        canonical = canonical.replace("And", "and").replace("Or", "or").replace(
            "Of", "of").replace("-Off", "-off").replace("-On", "-on")
        if canonical and canonical[0].islower():
            canonical = canonical[0].upper() + canonical[1:]
        if canonical.lower() not in seen:
            seen.add(canonical.lower())
            result.append(canonical)
    return result

# Reapply to all records
for record in data:
    record['target_diseases'] = normalize_diseases(record.get('target_diseases', []))

# Quick check
from collections import Counter
all_d = [d for r in data for d in r.get('target_diseases', [])]
lower_map = {}
for v in all_d:
    k = v.lower()
    lower_map.setdefault(k, [])
    lower_map[k].append(v)
dupes = {k: v for k, v in lower_map.items() if len(v) > 1}
print(f'Case variants remaining: {len(dupes)}')
print('Sample diseases:', list(Counter(all_d).keys())[:8])

Case variants remaining: 26
Sample diseases: ['Downy Mildew', 'Gray Mold', 'Purple Spot', 'Anthracnose', 'Leaf Spot', 'Wilt', 'Root Rot', 'Damping-off']


In [12]:
# ── CELL 6 — Normalize: target_pests ─────────────────────────────────────────
# Rule: Title Case — "red spider mites" → "Red Spider Mites"
# 43 case variants found

def normalize_pests(values):
    seen   = set()
    result = []
    for v in values:
        v = v.strip()
        if not v:
            continue
        canonical = v.title().replace("And", "and").replace("Or", "or")
        if canonical and canonical[0].islower():
            canonical = canonical[0].upper() + canonical[1:]
        if canonical.lower() not in seen:
            seen.add(canonical.lower())
            result.append(canonical)
    return result

for record in data:
    record['target_pests'] = normalize_pests(record.get('target_pests', []))

# Verify
sample = next(r for r in data if r['product_id'] == 'PN0001')
print('PN0001 pests:', sample['target_pests'])

PN0001 pests: ['Tea Caterpillar', 'Second Generation Cotton Bollworm', 'Pine Caterpillar', 'Rice Leafroller', 'Corn Borer', 'Diamondback Moth', 'Small Cabbage Moth']


In [13]:
# ── CELL 7 — Normalize: symptoms ─────────────────────────────────────────────
# Rule: lowercase — symptoms are descriptive phrases, not proper nouns
# "Yellowing" → "yellowing", "Dwarfing" → "dwarfing"

def normalize_symptoms(values):
    seen   = set()
    result = []
    for v in values:
        v = v.strip().lower()
        if not v:
            continue
        if v not in seen:
            seen.add(v)
            result.append(v)
    return result

for record in data:
    record['symptoms'] = normalize_symptoms(record.get('symptoms', []))

# Verify
sample = next(r for r in data if r['product_id'] == 'AF0001')
print('AF0001 symptoms:', sample['symptoms'])

AF0001 symptoms: ['yellowing', 'tip-drying', 'mosaic', 'dwarfing']


In [14]:
# ── CELL 8 — Normalize: active_ingredients ───────────────────────────────────
# Fix 7 case variants, preserve scientific names exactly

INGREDIENT_MAP = {
    'humic acid':                                          'Humic Acid',
    'plant essential oils':                                'Plant Essential Oils',
    'borneol':                                             'Borneol',
    'lavender':                                            'Lavender',
    'peppermint':                                          'Peppermint',
    'fluorine ion water':                                  'Fluorine Ion Water',
    'functional fertilizer containing medium and trace elements':
        'Functional Fertilizer Containing Medium and Trace Elements',
}

for record in data:
    record['active_ingredients'] = normalize_list(
        record.get('active_ingredients', []), INGREDIENT_MAP
    )

# Verify
sample = next(r for r in data if r['product_id'] == 'AF0001')
print('AF0001 ingredients:', sample['active_ingredients'])


AF0001 ingredients: ['Bacillus subtilis', 'Bacillus licheniformis', 'Bacillus pumilus', 'Bacillus gelatinous', 'Bacillus amyloliquefaciens']


In [15]:
# ── CELL 9 — Normalize: product_types ────────────────────────────────────────

PRODUCT_TYPE_MAP = {
    # Case fixes
    'microbial agent':                                     'Microbial Agent',
    'microbial fertilizer':                                'Microbial Fertilizer',
    'functional fertilizer containing medium and trace elements':
        'Functional Fertilizer Containing Medium and Trace Elements',
    'medium element water-soluble fertilizer':             'Medium Element Water-Soluble Fertilizer',
    'biological synergist':                                'Biological Synergist',
    'suspension concentrate':                              'Suspension Concentrate',
    'soluble concentrate':                                 'Soluble Concentrate',
    'emulsifiable concentrate':                            'Emulsifiable Concentrate',
    'wettable powder':                                     'Wettable Powder',

    # Semantic consolidation
    'agricultural adjuvant':                               'Adjuvant',
    'biological adjuvant':                                 'Adjuvant',
    'biological agricultural adjuvant':                    'Adjuvant',
    'acaricide synergist':                                 'Acaricide',
    'bio-insecticide':                                     'Insecticide',
    'organophosphorus insecticide':                        'Insecticide',
    'biological agent':                                    'Microbial Agent',
    'pure biological preparation':                         'Microbial Agent',
    'soluble liquid':                                      'Soluble Concentrate',
    'soluble liquid formulation':                          'Soluble Concentrate',
    'liquid formulation':                                  'Liquid',
    'liquid fertilizer':                                   'Liquid Fertilizer',
    'micronutrient fertilizer':                            'Micronutrient Water-Soluble Fertilizer',
    'water dispersible granule':                           'Water-Dispersible Granules',
    'soluble powder':                                      'Wettable Powder',
    'microemulsion':                                       'Emulsifiable Concentrate',
    'emulsion':                                            'Emulsifiable Concentrate',

    # New additions
    'aqueous solution':                                    'Soluble Concentrate',
    'microencapsulated suspension concentrate':            'Suspension Concentrate',
    'functional fertilizer':
        'Functional Fertilizer Containing Medium and Trace Elements',
    'micronutrient water-soluble fertilizer':              'Micronutrient Water-Soluble Fertilizer',
}

for record in data:
    record['product_types'] = normalize_list(
        record.get('product_types', []), PRODUCT_TYPE_MAP
    )

# Show result
from collections import Counter
all_types = [t for r in data for t in r.get('product_types', [])]
print('Product types after normalization:')
for t, c in Counter(all_types).most_common():
    print(f'  {c:>3}x  {t}')
print(f'\nTotal unique types: {len(Counter(all_types))}')

Product types after normalization:
   64x  Growth Promoter
   50x  Pesticide
   18x  Insecticide
   17x  Microbial Agent
   17x  Herbicide
   15x  Soluble Concentrate
    9x  Microbial Fertilizer
    9x  Adjuvant
    8x  Water-Soluble Fertilizer
    7x  Medium Element Water-Soluble Fertilizer
    7x  Suspension Concentrate
    7x  Emulsifiable Concentrate
    6x  Functional Fertilizer Containing Medium and Trace Elements
    6x  Wettable Powder
    4x  Macronutrient Water-Soluble Fertilizer
    4x  Biological Synergist
    3x  Liquid
    3x  Fungicide
    3x  Granules
    2x  Micronutrient Water-Soluble Fertilizer
    2x  Plant Growth Regulator
    2x  Acaricide
    1x  Liquid Fertilizer
    1x  Water-Dispersible Granules

Total unique types: 24


In [16]:
# ── CELL 10 — Normalize: dosages ─────────────────────────────────────────────
# Fix 2 case variants: mL vs ml

DOSAGE_MAP = {
    '30-40 ml/mu': '30-40 mL/mu',
    '30-50 ml/mu': '30-50 mL/mu',
}

for record in data:
    record['dosages'] = normalize_list(
        record.get('dosages', []), DOSAGE_MAP
    )

print('Dosage normalization done.')

Dosage normalization done.


In [17]:
# ── CELL 11 — Normalize: application_methods ─────────────────────────────────
# Fix 1 case variant

METHOD_MAP = {
    'stem and leaf spray': 'Stem and Leaf Spray',
}

for record in data:
    record['application_methods'] = normalize_list(
        record.get('application_methods', []), METHOD_MAP
    )

print('Application methods normalization done.')


Application methods normalization done.


In [18]:
# ── CELL 12 — Normalize: safety_warnings ─────────────────────────────────────
# Fix 1 case variant

WARNING_MAP = {
    'shelf life: 2 years': 'Shelf Life: 2 Years',
    'shelf life: 2 Years': 'Shelf Life: 2 Years',
}

for record in data:
    record['safety_warnings'] = normalize_list(
        record.get('safety_warnings', []), WARNING_MAP
    )

print('Safety warnings normalization done.')


Safety warnings normalization done.


In [21]:
# ── RELOAD AND REAPPLY ALL normalizations from scratch ───────────────────────
with open('product_entities_v2.json', encoding='utf-8') as f:
    data = json.load(f)

def normalize_diseases_final(values):
    seen   = set()
    result = []
    for v in values:
        v = v.strip()
        if not v:
            continue
        key = v.lower()
        if key not in seen:
            seen.add(key)
            canonical = v.title()
            canonical = canonical.replace("And", "and").replace("Or", "or").replace(
                "Of", "of").replace("-Off", "-off").replace("-On", "-on")
            if canonical and canonical[0].islower():
                canonical = canonical[0].upper() + canonical[1:]
            result.append(canonical)
    return result

for record in data:
    record['target_crops']        = normalize_crops(record.get('target_crops', []))
    record['target_diseases']     = normalize_diseases_final(record.get('target_diseases', []))
    record['target_pests']        = normalize_pests(record.get('target_pests', []))
    record['symptoms']            = normalize_symptoms(record.get('symptoms', []))
    record['active_ingredients']  = normalize_list(record.get('active_ingredients', []), INGREDIENT_MAP)
    record['product_types']       = normalize_list(record.get('product_types', []), PRODUCT_TYPE_MAP)
    record['dosages']             = normalize_list(record.get('dosages', []), DOSAGE_MAP)
    record['application_methods'] = normalize_list(record.get('application_methods', []), METHOD_MAP)
    record['safety_warnings']     = normalize_list(record.get('safety_warnings', []), WARNING_MAP)

# Check diseases only
from collections import Counter
all_d = [d for r in data for d in r.get('target_diseases', [])]
lower_map = {}
for v in all_d:
    lower_map.setdefault(v.lower(), []).append(v)
dupes = {k: v for k, v in lower_map.items() if len(v) > 1}
print(f'Disease case variants: {len(dupes)}')
if dupes:
    for k, v in list(dupes.items())[:5]:
        print(f'  {v}')
else:
    print('None ✓')
print('Sample:', list(Counter(all_d).keys())[:6])

Disease case variants: 26
  ['Downy Mildew', 'Downy Mildew', 'Downy Mildew', 'Downy Mildew', 'Downy Mildew', 'Downy Mildew', 'Downy Mildew', 'Downy Mildew', 'Downy Mildew', 'Downy Mildew', 'Downy Mildew', 'Downy Mildew', 'Downy Mildew', 'Downy Mildew']
  ['Gray Mold', 'Gray Mold', 'Gray Mold', 'Gray Mold', 'Gray Mold', 'Gray Mold', 'Gray Mold', 'Gray Mold', 'Gray Mold', 'Gray Mold']
  ['Purple Spot', 'Purple Spot', 'Purple Spot', 'Purple Spot']
  ['Anthracnose', 'Anthracnose', 'Anthracnose', 'Anthracnose', 'Anthracnose', 'Anthracnose', 'Anthracnose', 'Anthracnose', 'Anthracnose', 'Anthracnose', 'Anthracnose', 'Anthracnose', 'Anthracnose', 'Anthracnose']
  ['Leaf Spot', 'Leaf Spot', 'Leaf Spot', 'Leaf Spot', 'Leaf Spot', 'Leaf Spot', 'Leaf Spot']
Sample: ['Downy Mildew', 'Gray Mold', 'Purple Spot', 'Anthracnose', 'Leaf Spot', 'Wilt']


In [22]:
# ── CELL 13 — Verification ────────────────────────────────────────────────────
print('=== POST-NORMALIZATION VERIFICATION ===\n')

from collections import Counter

fields = ['target_crops', 'target_diseases', 'target_pests', 'symptoms',
          'active_ingredients', 'product_types', 'dosages',
          'application_methods', 'safety_warnings']

for field in fields:
    all_values = [v for r in data for v in r.get(field, [])]
    counter    = Counter(all_values)

    lower_map = {}
    for v in counter:
        key = v.lower().strip()
        lower_map.setdefault(key, [])
        lower_map[key].append(v)
    case_dupes = {k: v for k, v in lower_map.items() if len(v) > 1}

    status = 'WARN' if case_dupes else 'OK'
    print(f'  {status}  {field:<25} unique={len(counter):>3}  case_variants={len(case_dupes)}')
    if case_dupes:
        for k, variants in list(case_dupes.items())[:3]:
            print(f'       → {variants}')

print(f'\n  Total products: {len(data)} ✓')

=== POST-NORMALIZATION VERIFICATION ===

  OK  target_crops              unique=193  case_variants=0
  OK  target_diseases           unique= 67  case_variants=0
  OK  target_pests              unique=193  case_variants=0
  OK  symptoms                  unique=106  case_variants=0
  OK  active_ingredients        unique=102  case_variants=0
  OK  product_types             unique= 24  case_variants=0
  OK  dosages                   unique=146  case_variants=0
  OK  application_methods       unique= 46  case_variants=0
  OK  safety_warnings           unique=561  case_variants=0

  Total products: 114 ✓


In [23]:
# ── CELL 14 — Save ────────────────────────────────────────────────────────────
# Save JSON
with open('product_entities_normalized_v2.json', 'w', encoding='utf-8') as f:
    json.dump(data, f, ensure_ascii=False, indent=2)
print('Saved → product_entities_normalized_v2.json')

# Save CSV
entities_df = pd.DataFrame(data)
entities_df.to_csv('product_entities_normalized_v2.csv',
                   index=False, encoding='utf-8-sig')
print('Saved → product_entities_normalized_v2.csv')

# Verify
with open('product_entities_normalized_v2.json', encoding='utf-8') as f:
    verify = json.load(f)
print(f'Verified: {len(verify)} records ✓')

from google.colab import files
files.download('product_entities_normalized_v2.json')
files.download('product_entities_normalized_v2.csv')

print("""
=== PHASE 3 COMPLETE ✓ ===
Input:  product_entities_v2.json
Output: product_entities_normalized_v2.json
        product_entities_normalized_v2.csv
No API calls — 100% free
Next: notebook_04_retrieval_docs.ipynb
""")

Saved → product_entities_normalized_v2.json
Saved → product_entities_normalized_v2.csv
Verified: 114 records ✓


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


=== PHASE 3 COMPLETE ✓ ===
Input:  product_entities_v2.json
Output: product_entities_normalized_v2.json
        product_entities_normalized_v2.csv
No API calls — 100% free
Next: notebook_04_retrieval_docs.ipynb

